# Training & Ablation Grid — ResNet-18
### Multi-Domain Cross-Attention Fusion with Prototype Memory for Biomedical Image Forgery Localization

## Section 1: Imports, Paths, and Reproducibility

In [1]:
import subprocess
import sys

def _ensure_installed(pip_name, import_name=None):
    import_name = import_name or pip_name.replace("-", "_")
    try:
        __import__(import_name)
        print(f"{pip_name}: already available.")
    except ImportError:
        print(f"{pip_name}: not found, installing...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
        print(f"{pip_name}: installed.")

for _pkg in ["segmentation-models-pytorch", "albumentations"]:
    _ensure_installed(_pkg)

segmentation-models-pytorch: not found, installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00
segmentation-models-pytorch: installed.
albumentations: already available.


In [2]:
import os

# Must be set before torch loads, or the CUDA allocator is already
# initialised and ignores it — this alone saved 1-2GB on a T4.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import time
import json
import random
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device           : {device}")
if device.type == "cuda":
    print(f"GPU name         : {torch.cuda.get_device_name(0)}")
    print(f"GPU total memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected — check Settings > Accelerator before training.")

Device           : cuda
GPU name         : Tesla T4
GPU total memory : 15.6 GB


In [3]:
# Paths
# ================================
# EDIT SPLITS_INPUT_DIR to match the dataset you published from the Data
# Audit & Split notebook. Everything downstream (train/val/test loaders)
# depends on loading the exact same split that notebook produced.

DATA_DIR = Path("/kaggle/input/competitions/recodai-luc-scientific-image-forgery-detection")

SPLITS_INPUT_DIR = Path("/kaggle/input/notebooks/jennifersangwan/data-audit-canonical-split/analysis_outputs/splits")  # <-- EDIT THIS if you republish the audit notebook under a different name

# Optional: set this to a PREVIOUS run's published checkpoints dataset to
# resume interrupted configs or skip already-finished ones across sessions.
# Leave as None for a first run.
RESUME_INPUT_DIR = None  # e.g. Path("/kaggle/input/forgery-training-checkpoints")

OUTPUT_ROOT      = Path("/kaggle/working")
CHECKPOINT_DIR   = OUTPUT_ROOT / "checkpoints"
RESULTS_LOG_PATH = OUTPUT_ROOT / "results_log.csv"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# If a previous run's output was attached as input, seed this session's
# working directory from it BEFORE the config loop starts. This is what
# makes "already finished" and "resume from last epoch" work across
# separate Kaggle sessions rather than only within one open tab.
if RESUME_INPUT_DIR is not None and RESUME_INPUT_DIR.exists():
    prior_ckpt_dir = RESUME_INPUT_DIR / "checkpoints"
    prior_log_path = RESUME_INPUT_DIR / "results_log.csv"

    if prior_ckpt_dir.exists():
        for f in prior_ckpt_dir.glob("*.pt"):
            shutil.copy2(f, CHECKPOINT_DIR / f.name)
        print(f"Restored {len(list(prior_ckpt_dir.glob('*.pt')))} checkpoint file(s) from {prior_ckpt_dir}")

    if prior_log_path.exists() and not RESULTS_LOG_PATH.exists():
        shutil.copy2(prior_log_path, RESULTS_LOG_PATH)
        print(f"Restored results_log.csv from {prior_log_path}")
else:
    print("No RESUME_INPUT_DIR set — starting from a clean state.")

print(f"\nSPLITS_INPUT_DIR : {SPLITS_INPUT_DIR}")
print(f"CHECKPOINT_DIR   : {CHECKPOINT_DIR}")
print(f"RESULTS_LOG_PATH : {RESULTS_LOG_PATH}")

No RESUME_INPUT_DIR set — starting from a clean state.

SPLITS_INPUT_DIR : /kaggle/input/notebooks/jennifersangwan/data-audit-canonical-split/analysis_outputs/splits
CHECKPOINT_DIR   : /kaggle/working/checkpoints
RESULTS_LOG_PATH : /kaggle/working/results_log.csv


In [4]:
# Reproducibility
# ================================
# Every run gets its OWN call to set_all_seeds(seed) right before it starts
# (see Section 8) — not one global seed set here — because five configs
# share this notebook and each needs a clean, controlled seed state rather
# than inheriting whatever the previous config's training loop left behind.

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    # Without this, DataLoader worker processes each get their own
    # unseeded numpy/random state, so augmentation randomness inside
    # workers would NOT be controlled by set_all_seeds() above.
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print("Reproducibility helpers defined.")

Reproducibility helpers defined.


## Section 2: Load the Canonical Split

Loaded directly from the Data Audit & Split notebook's output — no `glob`, no re-deriving the
split here. Every one of the five configs in this notebook trains and evaluates on exactly these
same rows.

In [5]:
train_split_df = pd.read_csv(SPLITS_INPUT_DIR / "train_split.csv")
val_split_df   = pd.read_csv(SPLITS_INPUT_DIR / "val_split.csv")
test_split_df  = pd.read_csv(SPLITS_INPUT_DIR / "test_split.csv")

for name, df in [("train", train_split_df), ("val", val_split_df), ("test", test_split_df)]:
    print(f"{name:5s}: {len(df):4d} images")
    print(df["mask_size_group"].value_counts().to_dict())

# Cheap re-check: this cost nothing to verify and catches a mismatched or
# stale SPLITS_INPUT_DIR immediately rather than after a 40-epoch run.
train_ids = set(train_split_df["unique_id"])
val_ids   = set(val_split_df["unique_id"])
test_ids  = set(test_split_df["unique_id"])
assert not (train_ids & val_ids) and not (train_ids & test_ids) and not (val_ids & test_ids), \
    "Split overlap detected — check SPLITS_INPUT_DIR points at the right dataset version."
print("\nNo overlap between splits — safe to proceed.")

train: 1959 images
{'large': 691, 'medium': 533, 'micro': 448, 'small': 287}
val  :  420 images
{'large': 148, 'medium': 114, 'micro': 96, 'small': 62}
test :  420 images
{'large': 149, 'medium': 114, 'micro': 96, 'small': 61}

No overlap between splits — safe to proceed.


## Section 3: Canonical Mask Decoder, Dataset, and Transforms

`load_binary_mask()` here is the same function from the audit notebook, ported rather than
reimplemented — this is exactly the fix for the bug described above the code cells: the original
training notebook's own `squeeze` + `mask[0]` logic silently used the background channel for any
mask with 2+ channels. Every mask this notebook ever reads goes through this one function.

In [6]:
def load_binary_mask(mask_path):
    """
    Converts a stored mask into a proper 2D binary forgery mask.
    Ported from the Data Audit & Split notebook — same channel-detection
    logic, same convention (channel 0 = background, any positive in
    channels 1..end = forged), so training uses the same ground truth
    the split and the dataset statistics were built from.
    """
    mask = np.load(mask_path)

    if mask.ndim == 2:
        binary_mask = (mask > 0).astype(np.uint8)
    elif mask.ndim == 3:
        shape = mask.shape
        channel_axis = int(np.argmin(shape))
        mask_reordered = np.moveaxis(mask, channel_axis, 0)

        if mask_reordered.shape[0] == 1:
            binary_mask = (mask_reordered[0] > 0).astype(np.uint8)
        else:
            binary_mask = (np.any(mask_reordered[1:] > 0, axis=0)).astype(np.uint8)
    else:
        raise ValueError(f"Unexpected mask ndim={mask.ndim}, shape={mask.shape} for {mask_path}")

    return binary_mask


class ForgeryDataset(Dataset):
    """
    Loads forged image + binary mask pairs from a split dataframe (columns:
    image_path, mask_path). Returns image [3, 256, 256] and mask [1, 256, 256].
    """

    def __init__(self, split_df, transform=None):
        self.image_paths = split_df["image_path"].tolist()
        self.mask_paths  = split_df["mask_path"].tolist()
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img  = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        mask = load_binary_mask(self.mask_paths[idx])

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img  = augmented["image"]
            mask = augmented["mask"].float().unsqueeze(0)

        return img, mask


# Conservative augmentation on purpose: colour jitter would corrupt the
# noise stream (it reads camera noise fingerprints), fake JPEG artefacts
# would teach the frequency stream to detect the wrong thing, and Cutout
# could erase the forged region entirely. Flips, rotation, and small
# shift/scale are all spatially safe — the mask follows the image exactly.
train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=15, border_mode=0, p=0.70),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

BATCH_SIZE  = 4   # physical batch on GPU
ACCUM_STEPS = 4   # effective batch = 16
NUM_EPOCHS  = 40  # matches the original protocol — kept fixed for comparability

print("Datasets, transforms, and canonical mask decoder ready.")

Datasets, transforms, and canonical mask decoder ready.


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [7]:
def build_dataloaders(seed):
    """
    Fresh DataLoaders per run, with a seeded generator so shuffling order
    is controlled by that run's seed rather than whatever state the
    previous config left the global RNG in.
    """
    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(
        ForgeryDataset(train_split_df, train_transform),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
        pin_memory=True, prefetch_factor=2, persistent_workers=True,
        worker_init_fn=seed_worker, generator=g
    )
    val_loader = DataLoader(
        ForgeryDataset(val_split_df, val_transform),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
        pin_memory=True, persistent_workers=True
    )
    test_loader = DataLoader(
        ForgeryDataset(test_split_df, val_transform),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
        pin_memory=True, persistent_workers=True
    )
    return train_loader, val_loader, test_loader

print(f"Train batches per epoch (approx): {len(train_split_df) // BATCH_SIZE}")

Train batches per epoch (approx): 489


## Section 4: Domain Preprocessing — SRM Noise and FFT Frequency

Two fixed, zero-parameter signal extractors that give the noise and frequency streams a
genuinely different view of the image, rather than three copies of the same RGB signal learning
the same features. Unchanged from the original — this part of the architecture was never in
question, only its framing.

In [8]:
class SRMNoiseExtractor(nn.Module):
    """
    Extracts pixel-level noise residuals using Spatial Rich Model filters
    (Fridrich & Kodovsky, IEEE TIFS 2012). A copy-pasted region carries the
    noise fingerprint of its source — different camera, scanner, or
    compression — so SRM high-pass filters suppress real content and leave
    behind a noise texture that differs at forged boundaries.

    Three fixed 5x5 filters per colour channel, results averaged.
    Output: [B, 3, H, W] in [-1, 1]. Learnable params: 0.
    """

    def __init__(self):
        super().__init__()
        k1 = torch.tensor([
            [ 0,  0,  0,  0,  0], [ 0, -1,  2, -1,  0], [ 0,  2, -4,  2,  0],
            [ 0, -1,  2, -1,  0], [ 0,  0,  0,  0,  0]
        ], dtype=torch.float32) / 4.0

        k2 = torch.tensor([
            [-1,  2, -2,  2, -1], [ 2, -6,  8, -6,  2], [-2,  8,-12,  8, -2],
            [ 2, -6,  8, -6,  2], [-1,  2, -2,  2, -1]
        ], dtype=torch.float32) / 12.0

        k3 = torch.tensor([
            [ 0,  0,  0,  0,  0], [ 0,  0,  0,  0,  0], [ 0,  1, -2,  1,  0],
            [ 0,  0,  0,  0,  0], [ 0,  0,  0,  0,  0]
        ], dtype=torch.float32) / 2.0

        self.register_buffer('weight', torch.stack([k1, k2, k3]).unsqueeze(1))

    def forward(self, x):
        outs = []
        for c in range(3):
            outs.append(F.conv2d(x[:, c:c+1], self.weight, padding=2))
        out = torch.stack(outs).mean(0)
        return torch.clamp(out, -2, 2) / 2.0


class FFTMagnitudeExtractor(nn.Module):
    """
    Converts the image to frequency domain and extracts three radial
    magnitude bands. Copy-paste forgeries create hard spatial boundaries
    that show up as bursts of high-frequency energy — a pasted region
    breaks the frequency consistency of an otherwise clean scientific image.

    Output: [B, 3, H, W] in [0, 1]. Learnable params: 0.
    """

    def __init__(self, img_size=256):
        super().__init__()
        cx = cy = img_size // 2
        y = torch.arange(img_size).float() - cy
        x = torch.arange(img_size).float() - cx
        dist = (y.unsqueeze(1)**2 + x.unsqueeze(0)**2).sqrt()

        r_low = img_size // 8
        r_mid = img_size // 4

        self.register_buffer('low_mask',  (dist <= r_low).float()[None, None])
        self.register_buffer('mid_mask',  ((dist > r_low) & (dist <= r_mid)).float()[None, None])
        self.register_buffer('high_mask', (dist > r_mid).float()[None, None])

    def forward(self, x):
        gray = x.mean(dim=1, keepdim=True)
        fft = torch.fft.fftshift(torch.fft.fft2(gray, norm='ortho'))
        log_mag = torch.log(torch.abs(fft) + 1.0)

        out = torch.cat([
            log_mag * self.low_mask, log_mag * self.mid_mask, log_mag * self.high_mask
        ], dim=1)

        mn = out.amin(dim=[2, 3], keepdim=True)
        mx = out.amax(dim=[2, 3], keepdim=True)
        return (out - mn) / (mx - mn + 1e-8)


# sanity check before wiring into the full model
_srm = SRMNoiseExtractor().to(device)
_fft = FFTMagnitudeExtractor(img_size=256).to(device)
_dummy = torch.randn(2, 3, 256, 256).to(device)
print(f"SRM output : {tuple(_srm(_dummy).shape)}  learnable params: {sum(p.numel() for p in _srm.parameters())}")
print(f"FFT output : {tuple(_fft(_dummy).shape)}  learnable params: {sum(p.numel() for p in _fft.parameters())}")
del _srm, _fft, _dummy

SRM output : (2, 3, 256, 256)  learnable params: 0
FFT output : (2, 3, 256, 256)  learnable params: 0


## Section 5: Domain Adapters and Shared Encoder

One shared ResNet-18 encoder handles all three domain streams — three separate encoders would
cost ~3x the parameters and VRAM the T4 doesn't have to spare, and sharing forces the encoder to
learn features that generalise across RGB, noise, and frequency rather than overfitting to one.
Each stream gets a tiny adapter first because RGB, SRM, and FFT outputs sit in very different value
ranges, and ResNet's BatchNorm needs a consistent range to work with.

In [9]:
class DomainAdapter(nn.Module):
    """
    Normalises one domain stream's signal to [0, 1] before the shared
    encoder sees it. ~5K parameters — negligible next to the encoder.
    """

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.GELU(),
            nn.Conv2d(16, 3, kernel_size=1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


class SharedEncoder(nn.Module):
    """
    ResNet-18 pretrained on ImageNet, truncated after layer3.
    Returns main features [B, 256, 16, 16] and a [B, 64, 64, 64] skip
    connection from layer1 for the decoder.

    NOTE: ResNet-18 and ResNet-34 share identical channel widths at every
    stage (64/128/256/512) — only block counts differ. A ResNet-34 version
    of this notebook (Tier 4, if GPU quota allows) needs no other changes
    to this class beyond swapping which torchvision constructor is called.
    """

    def __init__(self):
        super().__init__()
        rn = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.stem   = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool)
        self.layer1 = rn.layer1
        self.layer2 = rn.layer2
        self.layer3 = rn.layer3

    def forward(self, x):
        x    = self.stem(x)
        skip = self.layer1(x)
        x    = self.layer2(skip)
        x    = self.layer3(x)
        return x, skip


# sanity check
_adapter = DomainAdapter().to(device)
_encoder = SharedEncoder().to(device)
_dummy   = torch.randn(2, 3, 256, 256).to(device)
_enc_out, _skip = _encoder(_adapter(_dummy))
print(f"Adapter params (one)  : {sum(p.numel() for p in _adapter.parameters()):,}")
print(f"Shared encoder params : {sum(p.numel() for p in _encoder.parameters())/1e6:.2f}M")
print(f"Encoder main output   : {tuple(_enc_out.shape)}")
print(f"Encoder skip output   : {tuple(_skip.shape)}")
del _adapter, _encoder, _dummy, _enc_out, _skip

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 159MB/s] 


Adapter params (one)  : 512
Shared encoder params : 2.78M
Encoder main output   : (2, 256, 16, 16)
Encoder skip output   : (2, 64, 64, 64)


## Section 6: Domain Stream Heads

Renamed from the original `AgentHead` to `DomainStreamHead` — same architecture, vocabulary
updated to match the paper's new title. Each stream gets its own confidence map: at a given spatial
position the noise stream might be far more confident than the frequency stream, so a fixed
equal-weight average would dilute a strong signal with a weak one.

In [10]:
class DomainStreamHead(nn.Module):
    """
    Per-stream feature refinement and spatial confidence estimation.
    Three instances in the full model — same architecture, separate weights.
    Input: [B, 256, 16, 16] shared encoder features.
    Output: feat [B, 128, 16, 16], conf [B, 1, 16, 16].
    """

    def __init__(self):
        super().__init__()
        self.refine = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.GELU()
        )
        self.conf_head = nn.Sequential(
            nn.Conv2d(128, 1, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        feat = self.refine(x)
        conf = self.conf_head(feat)
        return feat, conf


_head = DomainStreamHead().to(device)
_dummy = torch.randn(2, 256, 16, 16).to(device)
_feat, _conf = _head(_dummy)
print(f"Head output — feat: {tuple(_feat.shape)}  conf: {tuple(_conf.shape)}")
print(f"Params per head: {sum(p.numel() for p in _head.parameters()):,}")
del _head, _dummy, _feat, _conf

Head output — feat: (2, 128, 16, 16)  conf: (2, 1, 16, 16)
Params per head: 443,009


## Section 7: Cross-Domain Attention

Renamed from `CrossAgentAttention`/"debate" to `CrossDomainAttention` — this is the component
`use_attention` switches on or off in the ablation grid. Each stream's 16x16 feature map is flattened
to 256 tokens; all three streams' tokens (768 total) go through one joint multi-head self-attention
pass, so a noise anomaly and a frequency anomaly at different positions can still inform each other,
which a simple positional average could never do.

In [11]:
class CrossDomainAttention(nn.Module):
    """
    Three-way cross-domain self-attention.
    Input: fa, fb, fc each [B, 128, 16, 16]. Output: same shapes, enriched
    with evidence from all three streams.
    """

    def __init__(self, dim=128, num_heads=4, dropout=0.10):
        super().__init__()
        # one learnable embedding per stream so attention can tell which
        # stream a token came from — without it, RGB position (3,7) and
        # noise position (3,7) look identical to the attention mechanism
        self.domain_embeddings = nn.Parameter(torch.randn(3, dim) * 0.02)
        self.attention = nn.MultiheadAttention(
            embed_dim=dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 2, dim)
        )
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, fa, fb, fc):
        B, C, H, W = fa.shape
        N = H * W

        def flatten_and_tag(feat, idx):
            tokens = feat.flatten(2).transpose(1, 2)
            return tokens + self.domain_embeddings[idx]

        joint = torch.cat([
            flatten_and_tag(fa, 0), flatten_and_tag(fb, 1), flatten_and_tag(fc, 2)
        ], dim=1)

        normed = self.norm1(joint)
        attn_out, _ = self.attention(query=normed, key=normed, value=normed)
        joint = joint + attn_out
        joint = joint + self.ffn(self.norm2(joint))

        oa, ob, oc = joint.split(N, dim=1)

        def unflatten(tokens):
            return tokens.transpose(1, 2).view(B, C, H, W)

        return unflatten(oa), unflatten(ob), unflatten(oc)


_attn = CrossDomainAttention(dim=128, num_heads=4, dropout=0.10).to(device)
_fa = torch.randn(2, 128, 16, 16).to(device)
_fb = torch.randn(2, 128, 16, 16).to(device)
_fc = torch.randn(2, 128, 16, 16).to(device)
_oa, _ob, _oc = _attn(_fa, _fb, _fc)
print(f"Output shapes: {tuple(_oa.shape)}, {tuple(_ob.shape)}, {tuple(_oc.shape)}")
print(f"Params: {sum(p.numel() for p in _attn.parameters()):,}")
del _attn, _fa, _fb, _fc, _oa, _ob, _oc

Output shapes: (2, 128, 16, 16), (2, 128, 16, 16), (2, 128, 16, 16)
Params: 132,864


## Section 8: Prototype Memory Bank

Unchanged from the original — this component was already named accurately. This is what
`use_memory` switches on or off. During training it fills with embeddings of forged regions; at
inference it retrieves the most similar stored patterns and uses them to condition the current
prediction. Kept CPU-resident throughout, since it's updated with `no_grad` and would otherwise
just consume VRAM the three encoder passes need.

In [12]:
class SpatialPrototypeMemory(nn.Module):
    """CPU-resident forgery pattern bank with cross-attention retrieval."""

    def __init__(self, embed_dim=128, bank_size=256, top_k=8):
        super().__init__()
        self.embed_dim = embed_dim
        self.bank_size = bank_size
        self.top_k     = top_k

        self.register_buffer('bank',   torch.zeros(bank_size, embed_dim))
        self.register_buffer('ptr',    torch.tensor(0, dtype=torch.long))
        self.register_buffer('filled', torch.tensor(False))

        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.mem_attn   = nn.MultiheadAttention(
            embed_dim=embed_dim, num_heads=4, dropout=0.10, batch_first=True
        )
        self.norm = nn.LayerNorm(embed_dim)

    def _apply(self, fn):
        # .to()/.cuda() would otherwise move the bank too — force it back
        # to CPU immediately after any device move
        super()._apply(fn)
        self.bank   = self.bank.cpu()
        self.ptr    = self.ptr.cpu()
        self.filled = self.filled.cpu()
        return self

    @torch.no_grad()
    def update(self, features, masks):
        fh, fw = features.shape[2], features.shape[3]
        m_ds = F.interpolate(masks.float(), size=(fh, fw), mode='bilinear', align_corners=False)

        for i in range(features.shape[0]):
            m = m_ds[i, 0]
            if m.sum() < 1.0:
                continue
            f = features[i]
            emb = (f * m.unsqueeze(0)).sum(dim=[1, 2]) / (m.sum() + 1e-6)
            emb = F.normalize(emb, dim=0).cpu()

            idx = self.ptr.item() % self.bank_size
            self.bank[idx] = emb
            self.ptr += 1
            if not self.filled.item():
                self.filled = torch.tensor(True)

    def forward(self, features):
        if not self.filled.item():
            return features

        B, C, H, W = features.shape
        q = self.query_proj(features.mean(dim=[2, 3]))
        q_norm = F.normalize(q, dim=-1).detach().cpu()

        valid = min(self.ptr.item(), self.bank_size)
        bank_n = F.normalize(self.bank[:valid], dim=-1)
        sim = q_norm @ bank_n.t()

        k = min(self.top_k, valid)
        _, topk_idx = sim.topk(k, dim=-1)
        retrieved = self.bank[topk_idx].to(features.device)

        f_flat = features.flatten(2).transpose(1, 2)
        ctx, _ = self.mem_attn(query=f_flat, key=retrieved, value=retrieved)
        f_flat = self.norm(f_flat + ctx)

        return f_flat.transpose(1, 2).view(B, C, H, W)


_memory = SpatialPrototypeMemory(embed_dim=128, bank_size=256, top_k=8).to(device)
print(f"Bank device: {_memory.bank.device}  (must be cpu)")
_feat = torch.randn(4, 128, 16, 16).to(device)
_masks = torch.zeros(4, 1, 256, 256).to(device)
_masks[0, 0, 40:90, 50:110] = 1.0
_memory.update(_feat.detach(), _masks)
_out = _memory(_feat)
print(f"Bank entries after write: {_memory.ptr.item()}  |  output shape: {tuple(_out.shape)}")
del _memory, _feat, _masks, _out

Bank device: cpu  (must be cpu)
Bank entries after write: 1  |  output shape: (4, 128, 16, 16)


## Section 9: Decoder

Unchanged from the original. Takes the three (fused) stream features at 16x16 plus the 64x64
skip connection, and upsamples back to a full-resolution 256x256 logit map via two learned
transposed-convolution stages.

In [13]:
class ForgeryDecoder(nn.Module):
    """16x16 -> 64x64 -> 256x256 decoder with skip connection."""

    def __init__(self):
        super().__init__()
        self.stream_fuse = nn.Sequential(
            nn.Conv2d(128 * 3, 256, kernel_size=1, bias=False),
            nn.BatchNorm2d(256), nn.GELU()
        )
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=4),
            nn.BatchNorm2d(128), nn.GELU()
        )
        self.skip_fuse = nn.Sequential(
            nn.Conv2d(128 + 64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.GELU()
        )
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=4),
            nn.BatchNorm2d(64), nn.GELU()
        )
        self.pred_head = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1, bias=False),
            nn.GELU(),
            nn.Conv2d(32, 1, kernel_size=1)
        )

    def forward(self, stream_feats, skip):
        x = torch.cat(stream_feats, dim=1)
        x = self.stream_fuse(x)
        x = self.up1(x)
        x = torch.cat([x, skip], dim=1)
        x = self.skip_fuse(x)
        x = self.up2(x)
        return self.pred_head(x)


_decoder = ForgeryDecoder().to(device)
_feats = [torch.randn(2, 128, 16, 16).to(device) for _ in range(3)]
_skip  = torch.randn(2, 64, 64, 64).to(device)
_logit = _decoder(_feats, _skip)
print(f"Decoder output: {tuple(_logit.shape)}  params: {sum(p.numel() for p in _decoder.parameters()):,}")
del _decoder, _feats, _skip, _logit

Decoder output: (2, 1, 256, 256)  params: 994,657


## Section 10: MultiDomainFusionNet — the Ablatable Model

This is the model formerly called `AgenticForensicNet`. The architecture is identical to the
original — three domain streams, cross-domain attention, prototype memory, decoder — but
`use_attention` and `use_memory` are now constructor flags, and the modules they control are only
built (and only counted in the parameter total) when switched on. That's what makes the four fusion
configs in the grid below four dictionaries instead of four hand-maintained classes, and it's also
what fixes the params-table-vs-figure mismatch AI-2026 flagged: whatever config you build, the
printed parameter breakdown is the actual instantiated model, not a number copied into a table by
hand.

In [14]:
class MultiDomainFusionNet(nn.Module):
    """
    Multi-domain (RGB / noise / frequency) segmentation network with
    optional cross-domain attention and optional prototype memory.

    use_attention=False, use_memory=False:
        three domain streams, each processed independently, fused only by
        concatenation in the decoder — tests whether domain-specific
        preprocessing alone (without any cross-talk) helps at all.
    use_attention=True,  use_memory=False: cross-domain attention only.
    use_attention=False, use_memory=True:  prototype memory only.
    use_attention=True,  use_memory=True:  the full model.
    """

    def __init__(self, use_attention=True, use_memory=True):
        super().__init__()
        self.use_attention = use_attention
        self.use_memory    = use_memory

        self.srm = SRMNoiseExtractor()
        self.fft = FFTMagnitudeExtractor(img_size=256)

        self.rgb_adapter   = DomainAdapter()
        self.noise_adapter = DomainAdapter()
        self.freq_adapter  = DomainAdapter()

        self.encoder = SharedEncoder()

        self.rgb_head   = DomainStreamHead()
        self.noise_head = DomainStreamHead()
        self.freq_head  = DomainStreamHead()

        # Only instantiated when enabled — an ablated variant genuinely
        # has fewer parameters, not just an unused module sitting idle.
        self.cross_domain_attention = CrossDomainAttention(dim=128, num_heads=4, dropout=0.10) \
            if use_attention else None
        self.memory = SpatialPrototypeMemory(embed_dim=128, bank_size=256, top_k=8) \
            if use_memory else None

        self.decoder = ForgeryDecoder()

    def forward(self, x, masks=None):
        rgb_in   = self.rgb_adapter(x)
        noise_in = self.noise_adapter(self.srm(x))
        freq_in  = self.freq_adapter(self.fft(x))

        rgb_enc,   skip_r = self.encoder(rgb_in)
        noise_enc, skip_n = self.encoder(noise_in)
        freq_enc,  skip_f = self.encoder(freq_in)
        skip = (skip_r + skip_n + skip_f) / 3.0

        rgb_feat,   rgb_conf   = self.rgb_head(rgb_enc)
        noise_feat, noise_conf = self.noise_head(noise_enc)
        freq_feat,  freq_conf  = self.freq_head(freq_enc)

        # Cross-domain attention, only if enabled. When disabled, each
        # stream's features pass through untouched — no cross-talk at all.
        if self.use_attention:
            rgb_d, noise_d, freq_d = self.cross_domain_attention(rgb_feat, noise_feat, freq_feat)
            rgb_feat   = rgb_feat   + rgb_conf   * rgb_d
            noise_feat = noise_feat + noise_conf * noise_d
            freq_feat  = freq_feat  + freq_conf  * freq_d

        # Prototype memory, only if enabled.
        if self.use_memory:
            consensus = (rgb_feat + noise_feat + freq_feat) / 3.0
            if self.training and masks is not None:
                self.memory.update(consensus.detach(), masks)
            consensus = self.memory(consensus)
            rgb_feat   = rgb_feat   + consensus
            noise_feat = noise_feat + consensus
            freq_feat  = freq_feat  + consensus

        final_mask = self.decoder([rgb_feat, noise_feat, freq_feat], skip)
        return final_mask, rgb_conf, noise_conf, freq_conf


def count_params(module):
    return sum(p.numel() for p in module.parameters()) if module is not None else 0


def print_param_breakdown(model, label):
    print(f"\n{label} parameter breakdown:")
    print(f"  SRM / FFT (fixed)      : {count_params(model.srm):>10,}")
    print(f"  Domain adapters x3     : {count_params(model.rgb_adapter)*3:>10,}")
    print(f"  Shared encoder         : {count_params(model.encoder):>10,}")
    print(f"  Domain stream heads x3 : {count_params(model.rgb_head)*3:>10,}")
    print(f"  Cross-domain attention : {count_params(model.cross_domain_attention):>10,}"
          f"{'  (disabled)' if not model.use_attention else ''}")
    print(f"  Prototype memory       : {count_params(model.memory):>10,}"
          f"{'  (disabled)' if not model.use_memory else ''}")
    print(f"  Decoder                : {count_params(model.decoder):>10,}")
    total = sum(p.numel() for p in model.parameters())
    print(f"  TOTAL                  : {total:>10,}  ({total/1e6:.2f}M)")
    return total


# Build one instance of each fusion config to confirm everything wires
# together and to get real, printed parameter counts for the paper table
# before spending any GPU time training.
for _ua, _um, _label in [
    (False, False, "No attention, no memory"),
    (True,  False, "Attention only"),
    (False, True,  "Memory only"),
    (True,  True,  "Full model"),
]:
    _m = MultiDomainFusionNet(use_attention=_ua, use_memory=_um).to(device)
    print_param_breakdown(_m, _label)
    del _m
torch.cuda.empty_cache()


No attention, no memory parameter breakdown:
  SRM / FFT (fixed)      :          0
  Domain adapters x3     :      1,536
  Shared encoder         :  2,782,784
  Domain stream heads x3 :  1,329,027
  Cross-domain attention :          0  (disabled)
  Prototype memory       :          0  (disabled)
  Decoder                :    994,657
  TOTAL                  :  5,108,004  (5.11M)

Attention only parameter breakdown:
  SRM / FFT (fixed)      :          0
  Domain adapters x3     :      1,536
  Shared encoder         :  2,782,784
  Domain stream heads x3 :  1,329,027
  Cross-domain attention :    132,864
  Prototype memory       :          0  (disabled)
  Decoder                :    994,657
  TOTAL                  :  5,240,868  (5.24M)

Memory only parameter breakdown:
  SRM / FFT (fixed)      :          0
  Domain adapters x3     :      1,536
  Shared encoder         :  2,782,784
  Domain stream heads x3 :  1,329,027
  Cross-domain attention :          0  (disabled)
  Prototype memory 

## Section 11: Loss Function

In [15]:
class CombinedLoss(nn.Module):
    """Focal + Dice, equal weight — Focal handles the severe class imbalance,
    Dice pushes directly on IoU-correlated overlap."""

    def __init__(self, alpha=0.75, gamma=2.0, dice_weight=1.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.dice_weight = dice_weight
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, masks):
        probs = torch.sigmoid(logits)
        pt = torch.where(masks == 1, probs, 1 - probs)
        focal = self.alpha * (1.0 - pt) ** self.gamma * self.bce(logits, masks)

        pred = torch.sigmoid(logits)
        intersection = (pred * masks).sum(dim=(1, 2, 3))
        union = pred.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3))
        dice = 1 - (2 * intersection + 1e-6) / (union + 1e-6)

        return focal.mean() + self.dice_weight * dice.mean()

print("CombinedLoss defined.")

CombinedLoss defined.


## Section 12: The Ablation Grid

Five configs: the single-stream baseline (fairness anchor — same ResNet-18 backbone, same
training protocol, raw RGB only, via `segmentation_models_pytorch`), and the four
`MultiDomainFusionNet` fusion variants. All at `seed=42` for this first pass — Tier 3 (variance
across seeds) adds more rows to this list later, once Tier 1 shows which configs are worth
re-running with multiple seeds. That's a one-line addition when the time comes, not a rewrite.

In [16]:
CONFIGS = [
    {"name": "baseline_r18",                "model_type": "baseline",    "backbone": "resnet18", "use_attention": False, "use_memory": False, "seed": 42},
    {"name": "multidomain_r18_noattn_nomem","model_type": "multidomain", "backbone": "resnet18", "use_attention": False, "use_memory": False, "seed": 42},
    {"name": "multidomain_r18_attn_nomem",  "model_type": "multidomain", "backbone": "resnet18", "use_attention": True,  "use_memory": False, "seed": 42},
    {"name": "multidomain_r18_noattn_mem",  "model_type": "multidomain", "backbone": "resnet18", "use_attention": False, "use_memory": True,  "seed": 42},
    {"name": "multidomain_r18_full",        "model_type": "multidomain", "backbone": "resnet18", "use_attention": True,  "use_memory": True,  "seed": 42},
]

CONFIGS += [
    {**CONFIGS[0], "name": "baseline_r18_seed7", "seed": 7},
    {**CONFIGS[1], "name": "multidomain_r18_noattn_nomem_seed7", "seed": 7},
    {**CONFIGS[2], "name": "multidomain_r18_attn_nomem_seed7", "seed": 7},
    {**CONFIGS[3], "name": "multidomain_r18_noattn_mem_seed7", "seed": 7},
    {**CONFIGS[4], "name": "multidomain_r18_full_seed7", "seed": 7},

    {**CONFIGS[0], "name": "baseline_r18_seed123", "seed": 123},
    {**CONFIGS[1], "name": "multidomain_r18_noattn_nomem_seed123", "seed": 123},
    {**CONFIGS[2], "name": "multidomain_r18_attn_nomem_seed123", "seed": 123},
    {**CONFIGS[3], "name": "multidomain_r18_noattn_mem_seed123", "seed": 123},
    {**CONFIGS[4], "name": "multidomain_r18_full_seed123", "seed": 123},
]

for c in CONFIGS:
    print(c["name"])

baseline_r18
multidomain_r18_noattn_nomem
multidomain_r18_attn_nomem
multidomain_r18_noattn_mem
multidomain_r18_full
baseline_r18_seed7
multidomain_r18_noattn_nomem_seed7
multidomain_r18_attn_nomem_seed7
multidomain_r18_noattn_mem_seed7
multidomain_r18_full_seed7
baseline_r18_seed123
multidomain_r18_noattn_nomem_seed123
multidomain_r18_attn_nomem_seed123
multidomain_r18_noattn_mem_seed123
multidomain_r18_full_seed123


In [17]:
def build_model(config):
    if config["model_type"] == "baseline":
        model = smp.Unet(
            encoder_name=config["backbone"],
            encoder_weights="imagenet",
            in_channels=3,
            classes=1,
            activation=None
        )
    elif config["model_type"] == "multidomain":
        model = MultiDomainFusionNet(
            use_attention=config["use_attention"],
            use_memory=config["use_memory"]
        )
    else:
        raise ValueError("Unknown model_type: " + str(config["model_type"]))
    return model.to(device)


def forward_logits(model, images, masks, model_type):
    """Normalises the two model types' different return signatures down
    to just the logit tensor, so the training/eval loops don't need to
    branch on model_type themselves."""
    if model_type == "multidomain":
        logits, _, _, _ = model(images, masks)
        return logits
    else:
        return model(images)

print("Model factory ready.")

Model factory ready.


## Section 13: Results Log and Resume Helpers

In [18]:
RESULTS_COLUMNS = [
    "name", "model_type", "backbone", "use_attention", "use_memory", "seed",
    "params", "best_epoch", "best_val_iou", "test_iou", "test_dice",
    "total_epochs_run", "wall_clock_minutes"
]

def load_results_log():
    if RESULTS_LOG_PATH.exists():
        return pd.read_csv(RESULTS_LOG_PATH)
    return pd.DataFrame(columns=RESULTS_COLUMNS)

def is_config_complete(name):
    df = load_results_log()
    if len(df) == 0:
        return False
    row = df[df["name"] == name]
    return len(row) > 0 and pd.notna(row.iloc[0]["test_iou"])

def append_result_row(row_dict):
    df = load_results_log()
    df = df[df["name"] != row_dict["name"]]  # replace a partial/stale row for this config, if any
    df = pd.concat([df, pd.DataFrame([row_dict])], ignore_index=True)
    df.to_csv(RESULTS_LOG_PATH, index=False)

def find_resume_checkpoint(name):
    path = CHECKPOINT_DIR / f"{name}_last.pt"
    return path if path.exists() else None

print("Results log and resume helpers ready.")
print("\nAlready completed configs:")
_done = load_results_log()
print(_done[["name", "best_val_iou", "test_iou"]] if len(_done) > 0 else "  (none yet)")

Results log and resume helpers ready.

Already completed configs:
  (none yet)


## Section 14: Training and Evaluation Functions

**On resume:** the model, optimizer, scheduler, and scaler state are restored exactly, so
training continues correctly from where it stopped — the loss and gradients pick up as if never
interrupted. What is *not* preserved across a resume is the exact data-loading shuffle sequence
(that would need saving and restoring DataLoader worker RNG state, which isn't worth the complexity
here). This means a resumed run's batch order after the restart won't bit-for-bit match what an
uninterrupted run would have seen — it doesn't affect correctness, only exact reproducibility of a
resumed run's later epochs against a hypothetical uninterrupted twin.

In [19]:
def train_one_config(config, num_epochs=NUM_EPOCHS):
    name = config["name"]
    print("\n" + "=" * 70)
    print(f"Config: {name}")
    print("=" * 70)

    set_all_seeds(config["seed"])
    train_loader, val_loader, test_loader = build_dataloaders(config["seed"])

    model = build_model(config)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Params: {total_params/1e6:.2f}M")

    criterion = CombinedLoss(alpha=0.75, gamma=2.0, dice_weight=1.0).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-7)
    scaler = GradScaler(init_scale=2**16, growth_factor=2.0, backoff_factor=0.5, growth_interval=2000)

    start_epoch  = 0
    best_val_iou = 0.0
    best_epoch   = 0
    train_loss_history = []
    val_iou_history     = []

    resume_path = find_resume_checkpoint(name)
    if resume_path is not None:
        ckpt = torch.load(resume_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_epoch  = ckpt["epoch"]
        best_val_iou = ckpt["best_val_iou"]
        best_epoch   = ckpt["best_epoch"]
        train_loss_history = ckpt["train_loss_history"]
        val_iou_history     = ckpt["val_iou_history"]
        print(f"Resuming from epoch {start_epoch} (best so far: {best_val_iou:.4f} @ epoch {best_epoch})")

    if start_epoch >= num_epochs:
        print(f"{name} already trained for {start_epoch}/{num_epochs} epochs — skipping training.")

    wall_clock_start = time.time()

    for epoch in range(start_epoch, num_epochs):
        model.train()
        running_loss = 0.0
        num_batches  = 0
        epoch_start  = time.time()
        optimizer.zero_grad()

        for step, (images, masks) in enumerate(train_loader):
            images = images.to(device, non_blocking=True)
            masks  = masks.to(device, non_blocking=True).float()

            with autocast():
                logits = forward_logits(model, images, masks, config["model_type"])
                loss = criterion(logits, masks) / ACCUM_STEPS

            scaler.scale(loss).backward()
            running_loss += loss.item() * ACCUM_STEPS
            num_batches  += 1

            if (step + 1) % ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()

        scheduler.step()

        model.eval()
        val_iou_list = []
        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device, non_blocking=True)
                masks  = masks.to(device, non_blocking=True).float()
                with autocast():
                    logits = forward_logits(model, images, masks, config["model_type"])
                pred = (logits.sigmoid() > 0.5).float()
                intersection = (pred * masks).sum()
                union = pred.sum() + masks.sum() - intersection
                val_iou_list.append(((intersection + 1e-6) / (union + 1e-6)).item())

        val_iou  = float(np.mean(val_iou_list))
        avg_loss = running_loss / num_batches
        train_loss_history.append(avg_loss)
        val_iou_history.append(val_iou)

        epoch_time = time.time() - epoch_start
        print(
            f"[{name}] Epoch [{epoch+1:2d}/{num_epochs}]  Loss: {avg_loss:.4f}  |  "
            f"Val IoU: {val_iou:.4f}  |  Time: {epoch_time/60:.1f}min"
        )

        if val_iou > best_val_iou:
            best_val_iou = val_iou
            best_epoch   = epoch + 1
            torch.save({
                "epoch": epoch + 1, "model_state_dict": model.state_dict(),
                "val_iou": val_iou, "config": config
            }, CHECKPOINT_DIR / f"{name}_best.pt")
            print(f"  -> new best saved (Val IoU: {val_iou:.4f})")

        # Saved every epoch regardless of improvement — this is the resume point.
        torch.save({
            "epoch": epoch + 1, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_val_iou": best_val_iou, "best_epoch": best_epoch,
            "train_loss_history": train_loss_history, "val_iou_history": val_iou_history,
            "config": config
        }, CHECKPOINT_DIR / f"{name}_last.pt")

    wall_clock_minutes = (time.time() - wall_clock_start) / 60

    return {
        "model_type": config["model_type"], "total_params": total_params,
        "best_val_iou": best_val_iou, "best_epoch": best_epoch,
        "total_epochs_run": len(val_iou_history), "wall_clock_minutes": wall_clock_minutes,
        "val_loader": val_loader, "test_loader": test_loader
    }


@torch.no_grad()
def predict_with_tta(model, image, model_type):
    """Four-pass TTA: original, h-flip, v-flip, both — averaged after
    undoing each flip so predictions stay spatially aligned."""
    model.eval()
    predictions = []
    for flip_h, flip_v in [(False, False), (True, False), (False, True), (True, True)]:
        aug = image.clone()
        if flip_h: aug = torch.flip(aug, dims=[3])
        if flip_v: aug = torch.flip(aug, dims=[2])

        with torch.autocast(device_type="cuda"):
            logits = forward_logits(model, aug, None, model_type)
        prob = logits.sigmoid()

        if flip_h: prob = torch.flip(prob, dims=[3])
        if flip_v: prob = torch.flip(prob, dims=[2])
        predictions.append(prob)

    return torch.stack(predictions).mean(dim=0)


@torch.no_grad()
def evaluate_with_tta(model, loader, model_type, label="model"):
    """Runs TTA evaluation over a full loader. Returns mean IoU, mean Dice, IoU std."""
    model.eval()
    iou_list, dice_list = [], []
    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True).float()
        for i in range(images.shape[0]):
            avg_prob = predict_with_tta(model, images[i:i+1], model_type)
            pred = (avg_prob > 0.5).float()
            single_mask = masks[i:i+1]

            intersection = (pred * single_mask).sum()
            union = pred.sum() + single_mask.sum() - intersection
            iou_list.append(((intersection + 1e-6) / (union + 1e-6)).item())

            dice = (2.0 * intersection + 1e-6) / (pred.sum() + single_mask.sum() + 1e-6)
            dice_list.append(dice.item())

    print(f"  {label}: IoU {np.mean(iou_list):.4f} (+/-{np.std(iou_list):.4f})  Dice {np.mean(dice_list):.4f}")
    return float(np.mean(iou_list)), float(np.mean(dice_list)), float(np.std(iou_list))

print("Training and evaluation functions ready.")

Training and evaluation functions ready.


## Section 15: Run the Grid

Watch the very first epoch's printed time closely — that single number is your Tier-0 budget
check. Five configs at that per-epoch time x 40 epochs is your total estimate for this notebook; if
it clearly won't fit this session, it's fine to stop the cell after a config finishes (never mid-epoch
if avoidable) and re-run this notebook next session — finished configs are skipped, and the
in-progress one resumes from its last saved epoch.

In [20]:
for config in CONFIGS:
    name = config["name"]

    if is_config_complete(name):
        print(f"\n{name}: already complete (found in results_log.csv) — skipping.")
        continue

    result = train_one_config(config)

    print(f"\nEvaluating {name} on val and test sets with TTA...")
    best_ckpt = torch.load(CHECKPOINT_DIR / f"{name}_best.pt", map_location=device, weights_only=False)
    eval_model = build_model(config)
    eval_model.load_state_dict(best_ckpt["model_state_dict"])

    val_iou, val_dice, val_std   = evaluate_with_tta(eval_model, result["val_loader"],  config["model_type"], label=f"{name} (val)")
    test_iou, test_dice, test_std = evaluate_with_tta(eval_model, result["test_loader"], config["model_type"], label=f"{name} (test)")

    append_result_row({
        "name": name, "model_type": config["model_type"], "backbone": config["backbone"],
        "use_attention": config["use_attention"], "use_memory": config["use_memory"],
        "seed": config["seed"], "params": result["total_params"],
        "best_epoch": result["best_epoch"], "best_val_iou": val_iou,
        "test_iou": test_iou, "test_dice": test_dice,
        "total_epochs_run": result["total_epochs_run"],
        "wall_clock_minutes": round(result["wall_clock_minutes"], 1)
    })

    del eval_model
    torch.cuda.empty_cache()
    print(f"{name} complete and logged.\n")

print("\nAll configs in CONFIGS are either complete or have just finished.")


Config: baseline_r18


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Params: 14.33M


/tmp/ipykernel_58/1672145673.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=2**16, growth_factor=2.0, backoff_factor=0.5, growth_interval=2000)
/tmp/ipykernel_58/1672145673.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/1672145673.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


...

[multidomain_r18_attn_nomem_seed123] Epoch [ 1/40]  Loss: 0.9373  |  Val IoU: 0.2146  |  Time: 1.1min
  -> new best saved (Val IoU: 0.2146)
[multidomain_r18_attn_nomem_seed123] Epoch [ 2/40]  Loss: 0.8616  |  Val IoU: 0.2350  |  Time: 1.2min
  -> new best saved (Val IoU: 0.2350)
[multidomain_r18_attn_nomem_seed123] Epoch [ 3/40]  Loss: 0.8324  |  Val IoU: 0.2257  |  Time: 1.1min
[multidomain_r18_attn_nomem_seed123] Epoch [ 4/40]  Loss: 0.8120  |  Val IoU: 0.2496  |  Time: 1.1min
  -> new best saved (Val IoU: 0.2496)
[multidomain_r18_attn_nomem_seed123] Epoch [ 5/40]  Loss: 0.7877  |  Val IoU: 0.2348  |  Time: 1.1min
[multidomain_r18_attn_nomem_seed123] Epoch [ 6/40]  Loss: 0.7750  |  Val IoU: 0.2679  |  Time: 1.1min
  -> new best saved (Val IoU: 0.2679)
[multidomain_r18_attn_nomem_seed123] Epoch [ 7/40]  Loss: 0.7676  |  Val IoU: 0.2638  |  Time: 1.1min
[multidomain_r18_attn_nomem_seed123] Epoch [ 8/40]  Loss: 0.7542  |  Val IoU: 0.2414  |  Time: 1.1min
[multidomain_r18_attn_nomem_seed

/tmp/ipykernel_58/1672145673.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=2**16, growth_factor=2.0, backoff_factor=0.5, growth_interval=2000)
/tmp/ipykernel_58/1672145673.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/1672145673.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[multidomain_r18_noattn_mem_seed123] Epoch [ 1/40]  Loss: 0.9375  |  Val IoU: 0.2073  |  Time: 1.1min
  -> new best saved (Val IoU: 0.2073)
[multidomain_r18_noattn_mem_seed123] Epoch [ 2/40]  Loss: 0.8654  |  Val IoU: 0.2345  |  Time: 1.2min
  -> new best saved (Val IoU: 0.2345)
[multidomain_r18_noattn_mem_seed123] Epoch [ 3/40]  Loss: 0.8398  |  Val IoU: 0.2402  |  Time: 1.1min
  -> new best saved (Val IoU: 0.2402)
[multidomain_r18_noattn_mem_seed123] Epoch [ 4/40]  Loss: 0.8239  |  Val IoU: 0.2419  |  Time: 1.1min
  -> new best saved (Val IoU: 0.2419)
[multidomain_r18_noattn_mem_seed123] Epoch [ 5/40]  Loss: 0.8009  |  Val IoU: 0.2436  |  Time: 1.1min
  -> new best saved (Val IoU: 0.2436)
[multidomain_r18_noattn_mem_seed123] Epoch [ 6/40]  Loss: 0.7893  |  Val IoU: 0.2355  |  Time: 1.1min
[multidomain_r18_noattn_mem_seed123] Epoch [ 7/40]  Loss: 0.7829  |  Val IoU: 0.2624  |  Time: 1.1min
  -> new best saved (Val IoU: 0.2624)
[multidomain_r18_noattn_mem_seed123] Epoch [ 8/40]  Loss: 

/tmp/ipykernel_58/1672145673.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=2**16, growth_factor=2.0, backoff_factor=0.5, growth_interval=2000)
/tmp/ipykernel_58/1672145673.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/1672145673.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[multidomain_r18_full_seed123] Epoch [ 1/40]  Loss: 0.9344  |  Val IoU: 0.1960  |  Time: 1.1min
  -> new best saved (Val IoU: 0.1960)
[multidomain_r18_full_seed123] Epoch [ 2/40]  Loss: 0.8623  |  Val IoU: 0.2309  |  Time: 1.2min
  -> new best saved (Val IoU: 0.2309)
[multidomain_r18_full_seed123] Epoch [ 3/40]  Loss: 0.8285  |  Val IoU: 0.2438  |  Time: 1.1min
  -> new best saved (Val IoU: 0.2438)
[multidomain_r18_full_seed123] Epoch [ 4/40]  Loss: 0.8068  |  Val IoU: 0.2623  |  Time: 1.1min
  -> new best saved (Val IoU: 0.2623)
[multidomain_r18_full_seed123] Epoch [ 5/40]  Loss: 0.7860  |  Val IoU: 0.2522  |  Time: 1.1min
[multidomain_r18_full_seed123] Epoch [ 6/40]  Loss: 0.7716  |  Val IoU: 0.2611  |  Time: 1.1min
[multidomain_r18_full_seed123] Epoch [ 7/40]  Loss: 0.7653  |  Val IoU: 0.2717  |  Time: 1.1min
  -> new best saved (Val IoU: 0.2717)
[multidomain_r18_full_seed123] Epoch [ 8/40]  Loss: 0.7528  |  Val IoU: 0.2550  |  Time: 1.1min
[multidomain_r18_full_seed123] Epoch [ 9/4

## Results So Far

In [21]:
results_df = load_results_log()
display_cols = ["name", "use_attention", "use_memory", "params", "best_epoch", "best_val_iou", "test_iou", "test_dice", "wall_clock_minutes"]
results_df[display_cols] if len(results_df) > 0 else print("No results yet.")

,name,use_attention,use_memory,params,best_epoch,best_val_iou,test_iou,test_dice,wall_clock_minutes
0,baseline_r18,False,False,14328209,18,0.256547,0.275393,0.381171,42.5
1,multidomain_r18_noattn_nomem,False,False,5108004,19,0.220317,0.240108,0.338700,43.7
2,multidomain_r18_attn_nomem,True,False,5240868,30,0.238999,0.253122,0.354511,44.1
3,multidomain_r18_noattn_mem,False,True,5190820,19,0.224167,0.240840,0.336740,44.3
4,multidomain_r18_full,True,True,5323684,28,0.245566,0.260729,0.364708,45.1
5,baseline_r18_seed7,False,False,14328209,29,0.260565,0.279210,0.385665,42.2
6,multidomain_r18_noattn_nomem_seed7,False,False,5108004,11,0.224365,0.237713,0.336864,44.0
7,multidomain_r18_attn_nomem_seed7,True,False,5240868,17,0.225312,0.255809,0.356113,45.0
8,multidomain_r18_noattn_mem_seed7,False,True,5190820,20,0.211952,0.228855,0.322931,44.9
9,multidomain_r18_full_seed7,True,True,5323684,25,0.223591,0.249154,0.348468,45.6
